# Qwen 3.5 4B QLoRA Training

Train a reasoning base model from 1,320 curated examples.

**Steps:**
1. Check GPU
2. Install unsloth
3. Upload training data
4. Train
5. Download merged model

In [ ]:
# Step 1: Check GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# Step 2: Install unsloth
!pip install unsloth 2>&1 | tail -5
print("Install done!")

In [ ]:
# Step 3: Upload training data
# Click the file icon on the left sidebar, then upload:
#   claude_reasoning.jsonl (from agents/distill/data/)
#
# Or run this cell to upload via dialog:
from google.colab import files
uploaded = files.upload()  # select claude_reasoning.jsonl
print(f"Uploaded: {list(uploaded.keys())}")

In [ ]:
# Step 4: Train
import json
import torch
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import Dataset

print(f"GPU: {torch.cuda.get_device_name()}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

# Load data
examples = []
with open("claude_reasoning.jsonl", "r") as f:
    for line in f:
        if line.strip():
            examples.append(json.loads(line))
print(f"Loaded {len(examples)} examples")

# Load model
print("Loading Qwen 3.5 4B...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen3.5-4B",
    max_seq_length=1024,
    dtype=None,
    load_in_4bit=True,
)

# QLoRA
print("Applying QLoRA...")
model = FastLanguageModel.get_peft_model(
    model,
    r=16, lora_alpha=16, lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",
)
print(f"VRAM after QLoRA: {torch.cuda.memory_allocated()/1024**3:.1f} GB")

# Format dataset
dataset = Dataset.from_list(examples)
def format_chat(example):
    return {"text": tokenizer.apply_chat_template(
        example["messages"], tokenize=False, add_generation_prompt=False)}
dataset = dataset.map(format_chat, remove_columns=dataset.column_names)

# Train
args = TrainingArguments(
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    warmup_ratio=0.03,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=5,
    output_dir="./checkpoints",
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    save_strategy="epoch",
    report_to="none",
)

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    args=args, train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=1024, packing=False,
)

from unsloth import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

print("Training...")
stats = trainer.train()
print(f"\nDone! Loss: {stats.training_loss:.4f}")
print(f"Peak VRAM: {torch.cuda.max_memory_allocated()/1024**3:.1f} GB")

In [ ]:
# Step 5: Save merged model
print("Merging LoRA...")
model.save_pretrained_merged(
    "merged_4b", tokenizer, save_method="merged_16bit"
)
print("Saved to merged_4b/")

# Also try GGUF export
try:
    print("Exporting GGUF Q5_K_M...")
    model.save_pretrained_gguf(
        "merged_4b_gguf", tokenizer, quantization_method="q5_k_m"
    )
    print("GGUF saved to merged_4b_gguf/")
except Exception as e:
    print(f"GGUF export failed: {e}")
    print("Download FP16 and convert locally instead.")

In [ ]:
# Step 6: Download the model
# Option A: Zip and download directly
!tar czf merged_4b.tar.gz merged_4b/
from google.colab import files
files.download("merged_4b.tar.gz")

# Option B: If too large, save to Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r merged_4b /content/drive/MyDrive/